In [ ]:
import numpy as np
import pydicom
from pydicom.uid import ExplicitVRLittleEndian
from pydicom.dataset import FileMetaDataset


# =========================
# User inputs
# =========================
npy_path = "cbct_projections.npy"          # shape: (N, rows, cols)
template_dcm_path = "template_proj.dcm"    # single projection DICOM
output_dcm_path = "cbct_projections.dcm"


# =========================
# Load data
# =========================
proj = np.load(npy_path)                   # (num_frames, rows, cols)
ds = pydicom.dcmread(template_dcm_path)


# =========================
# Validate projection stack
# =========================
if proj.ndim != 3:
    raise ValueError(
        f"Expected 3D array (num_frames, rows, cols), got {proj.shape}"
    )

num_frames, rows, cols = proj.shape
print(f"Loaded projections: {num_frames} frames, size {rows} x {cols}")


# =========================
# Convert datatype
# =========================
# CBCT projection DICOMs must be integer pixel data
proj = proj.astype(np.int16)


# =========================
# Update core DICOM image tags
# =========================
ds.Rows = rows
ds.Columns = cols
ds.NumberOfFrames = num_frames

ds.SamplesPerPixel = 1
ds.PhotometricInterpretation = "MONOCHROME2"

ds.BitsAllocated = 16
ds.BitsStored = 16
ds.HighBit = 15
ds.PixelRepresentation = 1  # 1 = signed, 0 = unsigned


# =========================
# Replace pixel data
# =========================
# Multi-frame DICOM expects all frames concatenated
ds.PixelData = proj.tobytes()


# =========================
# Optional CBCT / projection hints
# =========================
# Only set these if they are appropriate for your system
ds.ImageType = ["ORIGINAL", "PRIMARY", "PROJECTION"]
ds.Modality = ds.get("Modality", "DX")  # keep original if present

# If needed to avoid bad scaling in viewers
ds.RescaleSlope = 1
ds.RescaleIntercept = 0


# =========================
# File meta & transfer syntax
# =========================
ds.file_meta = FileMetaDataset()
ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

ds.is_little_endian = True
ds.is_implicit_VR = False


# =========================
# Save output DICOM
# =========================
ds.save_as(output_dcm_path)
print(f"✅ Saved multi-frame CBCT DICOM: {output_dcm_path}")


# =========================
# Sanity check
# =========================
ds_check = pydicom.dcmread(output_dcm_path)
print("Verification:")
print("  NumberOfFrames:", ds_check.NumberOfFrames)
print("  pixel_array shape:", ds_check.pixel_array.shape)
``